In [1]:
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset -p data --unzip

Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0
100%|█████████████████████████████████████| 41.0M/41.0M [00:04<00:00, 9.89MB/s]



In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Download necessary NLTK data
nltk.download('stopwords')

# LOAD DATA

fake = pd.read_csv('./data/Fake.csv')
true = pd.read_csv('./data/True.csv')

fake['label'] = 1
true['label'] = 0

df = pd.concat([fake, true], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

df = df[['text', 'label']]

# PREPROCESSING FUNCTION
ps = PorterStemmer()

def prepare_text(text):
    text = str(text)

    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    stop_words = set(stopwords.words('english'))
    words = [ps.stem(word) for word in words if word not in stop_words]
    
    return ' '.join(words)

def clean_text(tekst):
    tekst = str(tekst)

    tekst = re.sub(r"^\s*[A-Za-z .,'/-]{0,60}\(Reuters\)\s*-\s*", "", tekst)

    tekst = re.sub(r"\(reuters\)", " ", tekst, flags=re.IGNORECASE)
    tekst = re.sub(r"\breuters\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"(featured image via|image via|featured image|getty images|pic\.twitter\.com|screen capture|screenshot via|photo by)", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"(https?://\S+|www\.\S+|\b\S+\.com\b)", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"@\w+", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\s+", " ", tekst)
    tekst = tekst.strip()

    return tekst



print("Cleaning text... (this may take a few minutes)")
df['text'] = df['text'].apply(prepare_text)
df["text"] = df["text"].apply(clean_text)
print("Cleaning complete!")

df["length"] = df["text"].str.len()
df = df[df["length"] >= 40]
df = df.reset_index(drop=True)

df.to_pickle("./data/cleaned_news_data.pkl")

[nltk_data] Downloading package stopwords to /home/uno/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Cleaning text... (this may take a few minutes)
Cleaning complete!
